In [0]:
spark

In [0]:
# display(dbutils.fs.ls("/databricks-datasets/airlines/"))

In [0]:
# %fs ls /databricks-datasets

In [0]:
# dbutils.fs.ls("dbfs:/databricks-datasets/airlines")


## finding file type 

In [0]:
%pip install python-magic
import magic

path = "/dbfs/databricks-datasets/airlines/part-00000"

file_type = magic.from_file(path, mime=True)
print(f"Detected MIME type: {file_type}")


### Reading of .md file 

In [0]:
md_path = "dbfs:/databricks-datasets/airlines/README.md"
content = dbutils.fs.head(md_path)  # reads first 10,000 bytes
print(content)

In [0]:
# df = spark.read.format("csv").option("inferschema",True).load("dbfs:/databricks-datasets/airlines")

In [0]:
# display(df)

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/sample/json/"))

In [0]:
amazon_df = spark.read.format("json").option("inferschema",True).load("dbfs:/databricks-datasets/nyctaxi/sample/json/")
display(amazon_df)


In [0]:
data = [1, 2, 3, 4, 5, 6]
rdd = spark.sparkContext.parallelize(data)
rdd.collect()



## Rdd 

In [0]:
from pyspark.sql import SparkSession
data = ["apple","banana","grape"]
rdd = spark.sparkContext.parallelize(data)
upper_rdd = rdd.map(lambda x: x.upper())
print(upper_rdd.collect())

In [0]:
from pyspark.sql import SparkSession

data = ["apple", "banana", "grape"]
df = spark.createDataFrame(data, "string").toDF("fruit")
upper_df = df.selectExpr("upper(fruit) as fruit_upper")
display(upper_df)

In [0]:
# Coded in PySpark

# Let's assume we already have a DataFrame with an integer column 'price'
# For example:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("LazyEvalExample").getOrCreate()
data = [(10,), (20,), (30,)]
df = spark.createDataFrame(data, ["price"])

# Transformation 1: Multiply price by 2
df = df.withColumn('price', col('price') * 2)

# Transformation 2: Multiply price by 3
df = df.withColumn('price', col('price') * 3)

# Transformation 3: Multiply price by 5
df = df.withColumn('price', col('price') * 5)

# No execution has happened yet! (Lazy Evaluation)

# This action triggers the execution
df_collect = df.collect()

# Output: [Row(price=300), Row(price=600), Row(price=900)]
print(df_collect)


# b35&b36 coding starts

### csv files path: https://github.com/datablist/sample-csv-files?tab=readme-ov-file


##  Create DataFrame from Dictionary: 

In [0]:
data_dict = [{"ID": 1, "Name": "Alice"}, {"ID": 2, "Name": "Bob"}] 
df_from_dict = spark.createDataFrame(data_dict) 
df_from_dict.show() 

##  Create Empty DataFrame

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
schema = StructType([ 
                     StructField("id", IntegerType(), True),
                     StructField("name", StringType(), True)
                     ])

In [0]:
empty_df = spark.createDataFrame([],schema)
display(empty_df)

## 5. Creating DataFrame from Structured Data (CSV, JSON, Parquet

In [0]:
df_csv = spark.read.csv("/Volumes/manu/default/csv_file", header=True, 
inferSchema=True) 
df_csv.display()

In [0]:
# df_csv.printSchema()

###Show the first 3 rows, truncate columns to 25 characters, and display vertically: 

In [0]:
df_csv.show(n=3, truncate=25, vertical=True)

###Show the first 10 rows:

In [0]:
df_csv.show(10) 

## Loading Data from CSV File into a DataFrame

1.Import Required Libraries 

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType,StringType, DoubleType 

2. Define the Schema

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("Index", IntegerType(), True),
    StructField("Customer Id", StringType(), True),
    StructField("First Name", StringType(), True),
    StructField("Last Name", StringType(), True),
    StructField("Company", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Phone 1", StringType(), True),
    StructField("Phone 2", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("Subscription Date", DateType(), True),
    StructField("Website", StringType(), True)
])


3. Read the CSV File 

In [0]:
df = spark.read.csv("/Volumes/manu/default/csv_file", schema=schema, 
header=True) 

In [0]:
display(df.limit(10))

### 4. Load Multiple CSV Files

Load Multiple CSV Files with diffrent schema 

 Ensure that the schema is consistent across all files. 

In [0]:
file_paths = ["/Volumes/manu/default/csv_file/customers-100.csv", "/Volumes/manu/default/csv_file/leads-100.csv", "/Volumes/manu/default/csv_file/people-100.csv"]

In [0]:
df = spark.read.csv(file_paths, header=True, inferSchema=True)

If any file:

Has extra columns → they will be ignored.

Has missing columns → the missing values will be null.

In [0]:
display(df.limit(10))

Load Multiple CSV Files with same schema

In [0]:
file_paths = ["/Volumes/manu/default/csv_file/customers-100.csv", "/Volumes/manu/default/csv_file/customers-1000.csv"]

In [0]:
df = spark.read.csv(file_paths, header=True, inferSchema=True)

In [0]:
display(df.limit(10))

Interview question:
-  How Does inferSchema Work?
- Ans:-Behind the Scenes: When you use inferSchema, Spark runs a job that scans the CSV file from top to bottom to identify the best-suited data type for each column based on the values it encounters.

Does It Make Sense to Use inferSchema?
- Pros: 
 Useful when the schema of the file keeps changing, as it allows Spark to 
automatically detect the data types.

- Cons: 
- Performance Impact: Spark must scan the entire file, which can take extra 
time, especially for large files. 
- Loss of Control: You lose the ability to explicitly define the schema, which may 
lead to incorrect data types if the data is inconsistent.

### Defining Schema as a String

In [0]:
customerSchema = '''
    Index Integer,
    `Customer Id` String,
    `First Name` String,
    `Last Name` String,
    Company String,
    City String,
    Country String,
    `Phone 1` String,
    `Phone 2` String,
    Email String,
    `Subscription Date` Date,
    Website String
'''



Load the DataFrame with the defined schema

In [0]:
df = spark.read.load("/Volumes/manu/default/csv_file/customers-100.csv", format="csv", schema=customerSchema, header=True)

In [0]:
df.printSchema()

-  **Schema Definition**: Both methods define a schema for the DataFrame,accommodating the dataset's requirements, including handling null values where applicable. 
- **Data Types**: The Joining_Date column is defined as StringType to accommodate potential date format issues or missing values. 
- **Loading the DataFrame**: The spark.read.load method is used to load the CSV file into a DataFrame using the specified schema. 
- **Printing the Schema**: The df.printSchema() function allows you to verify that the DataFrame is structured as intended. 

## PySpark Column Selection & Manipulation: Key Techniques

. Different Methods to Select Columns 
- Using col() function/ column() / string way: 



In [0]:
from pyspark.sql.functions import *

In [0]:
#Using col() function 
df.select(col("First Name")).show(10)

In [0]:
#Using column() function 
df.select(column("City")).show(10)

In [0]:
#Directly using string name 
df.select("Website").show()

2. Selecting Multiple Columns Together 

In [0]:
df2 = df.select("Customer Id", "First Name", col("City"), column("Website"), 
df.Email) 
df2.show(10) 

In [0]:
x%md
3. Listing All Columns in a DataFrame 

In [0]:
df.columns

4.Renaming Columns with alias() 

In [0]:
df.select( 
  col("First Name").alias('EmployeeName'),  # Rename "First Name" to "EmployeeName" 
  col("Last Name").alias('L_name'),  # Rename "Last Name" to L_name" 
  column("Company"),  # Select "Company" 
  #df.Subscription Date  # Select "Subscription Date" cannot be done because it has " " between words
  df["Subscription Date"]
).show(10) 

5. Using selectExpr() for Concise Column Selection

selectExpr() allows you to use SQL expressions directly and rename columns 

In [0]:
df.selectExpr("`First Name` as EmployeeName", "`Last Name` as Last_Name","Email as Mail","Company").show(10) 

**Summary **
-  Use col(), column(), or string names to select columns. 
-  Use expr() and selectExpr() for SQL-like expressions and renaming. 
- Use alias() to rename columns. 
- Get the list of columns using df.columns. 

## PySpark DataFrame Manipulation part 2: Adding, Renaming, and Dropping Columns 

1. Adding New Columns with withColumn()

In [0]:
#Add a constant value column:
newdf = df.withColumn("NewColumn", lit(1))

In [0]:
# Add a column based on an expression: 
newdf = df.withColumn("withinCountry", expr("Country == 'India'"))

In [0]:
# display(newdf)

**summery**
- This function allows adding multiple columns, including calculated ones: 
- Example: 
  - Assign a constant value with lit(). 
  - Perform calculations using existing columns like multiplying values. 

2. Renaming Columns with withColumnRenamed() 

PySpark provides the withColumnRenamed() method to rename columns. This is especially useful when you want to change the names for clarity or to follow naming conventions: 

In [0]:
new_df = df.withColumnRenamed("Email", "mail")

In [0]:
#• Handling column names with special characters or spaces: If a column has specialcharacters or spaces, you need to use backticks (`) to escape it:
newdf.select("`First Name`").show(5) 

3. Dropping Columns with drop()

To remove unwanted columns, you can use the drop() method:

In [0]:
df2 = df.drop("Country")

In [0]:
df2.limit(5).display()

In Spark, DataFrames are immutable by nature. This means that after creating a DataFrame,its contents cannot be changed. All transformations like adding, renaming, or droppingcolumns result in a new DataFrame, keeping the original one intact. 
-  For instance, dropping columns creates a new DataFrame without altering the original:

In [0]:
newdf = df.drop("First Name", "Country")

In [0]:
newdf.limit(5).display()

-- **_Key Points_** 
-   Use withColumn() for adding columns, with lit() for constant values and expressionsfor computed values. 
-  Use withColumnRenamed() to rename columns and backticks for special characters or spaces. 
-   Use drop() to remove one or more columns. 
-   DataFrames are immutable in Spark—transformations result in new DataFrames,leaving the original unchanged

##  changing data types, filtering data, and handling unique/distinct values in PySpark

1.Changing Data Types (Schema Transformation)


In PySpark, you can change the data type of a column using the cast() method. This is helpful when you need to convert data types.

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col
# Change the 'Index' column from integer to double 
df = df.withColumn("Index", col("Index").cast("double")) 

In [0]:
df.printSchema()

In [0]:
#HW
# Convert 'Subscription Date' column to string 
df2 = df.withColumn("Subscription Date", col("Subscription Date").cast("string"))
df2.printSchema() 

2. Filtering Data 

You can filter rows based on specific conditions. For instance, to filter **Index**  greater than 90

In [0]:
filtered_df = df.filter(col("Index") > 90) 
filtered_df.show() 

3. Multiple Filters (Chaining Conditions) 

You can also apply multiple conditions using & or | (AND/OR) to filter data.

In [0]:
filtered_df = df.filter((df["Index"] < 30) & (df["City"] == "Isabelborough"))
filtered_df.show()

4. Filtering on Null or Non-Null Values

Filtering based on whether a column has NULL values or not is crucial for data cleaning

In [0]:
df.display()

# Filter rows where 'Address' is NULL 


In [0]:
# using this code to create null values 
df3 = df.withColumn("City", when(col("Index") > 98, None).otherwise(col("City")))

In [0]:
display(df3)

In [0]:
# Filter rows where 'City' is NULL
filtered_df = df3.filter(df["City"].isNull()) 
filtered_df.show()

In [0]:
## HW
# Filter rows where 'Country' is NOT NULL 
filtered_df = df.filter(df["City"].isNotNull()) 
filtered_df.show(10)

## Handling Unique or Distinct Data 

In [0]:
# Get distinct rows from the entire DataFrame
unq_df  = df.distinct()
unq_df.show()

In [0]:
# Get distinct values from the 'City' column
unique_City_df = df.select("City").distinct() 
unique_City_df.show() 

To remove duplicates based on specific columns, such as Email or Phone, use dropDuplicates(): 

In [0]:
# Remove duplicates based on 'Email' column 
unique_df = df.dropDuplicates(["Email"]) 
unique_df.show()

In [0]:
df.printSchema()

In [0]:
# Remove duplicates based on both 'Phone' and 'Email' 
unique_df = df.dropDuplicates(["Phone 1", "Email"]) 
unique_df.show() 

## 6. Counting Distinct Values

In [0]:
# Count distinct values in the 'City' column 
distinct_count_City = df.select("City").distinct().count() 
print("Distinct City Count:", distinct_count_City)

In [0]:
# HW
# Count distinct combinations of 'City' and'country'
City_combinations_count = df.select("City","Country").distinct().count()
print("Distinct City and Country Combinations:",
City_combinations_count)

%md
1. Changing Data Types: Easily modify column types using .cast(). E.g., change 'Salary' to double or 'Phone' to string for better data handling. 
2. Filtering Data: Use .filter() or .where() to extract specific rows. For example, filteremployees with a salary over 50,000 or non-null Age. 
3. Multiple Conditions: Chain filters with & and | to apply complex conditions, such as finding employees over 30 in the IT department. 
4. Handling NULLs: Use .isNull() and .isNotNull() to filter rows with missing or available values, such as missing addresses or valid emails. 
5. Unique/Distinct Values: Use .distinct() to get unique rows or distinct values in a column. Remove duplicates based on specific fields like Email or Phone using.dropDuplicates(). 
6. Count Distinct Values: Count distinct values in one or multiple columns to analyze data diversity, such as counting unique departments or combinations of Department and Performance_Rating. 

## SortingAndStringFunctions

In [0]:
# Sample data 
data = [ 
    ("USA", "North America", 100, 50.5), 
    ("India", "Asia", 300, 20.0), 
    ("Germany", "Europe", 200, 30.5), 
    ("Australia", "Oceania", 150, 60.0), 
    ("Japan", "Asia", 120, 45.0), 
    ("Brazil", "South America", 180, 25.0) 
] 
 
# Define the schema 
columns = ["Country", "Region", "UnitsSold", "UnitPrice"] 
 
# Create DataFrame 
df = spark.createDataFrame(data, columns) 
 
# Display the original DataFrame 
df.show() 

### Sorting the DataFrame

1. Sort by a single column
(ascending order): 

Note: By default, the sorting isin ascending order. This shows the top 5 countries in alphabetical order. 

In [0]:
df.orderBy("Country").show(5)

2. Sort by multiple columns: 

Note: Here, the DataFrame is sorted first by Country (ascending), and within the same country, it is sorted by UnitsSold in ascending order. 

In [0]:
df.orderBy("Country","UnitsSold").show()

3. Sort by a column in descending order and limit: 

Note: This sorts the DataFrame by Country in descending order and limits the output to the top 3 rows. 

In [0]:
sorted_df = df.orderBy(desc("Country")).limit(3)
sorted_df.show()

4. Sorting with null values last:

Note: This ensures that null values (if present) are placed at the end when sorting by Country.

In [0]:
sorted_df = df.orderBy(col("Country").desc(),nulls_last=True).show(5)

- **Summary of Key Functions:** 
-  Sorting: You can sort a DataFrame by one or more columns using .orderBy() or .sort().By default, sorting is ascending, but you can change it using asc() or desc(). 
- These functions and transformations are common in PySpark for manipulating and querying data effectively! 


### String Functions 

 1. Convert the first letter of each word to uppercase (initcap):

In [0]:
df.select(initcap(col("Country"))).show()

Note: This transforms the first letter of each word in the Country column to uppercase.

2. Convert all text to lowercase (lower):

In [0]:
df.select(lower(col("Country"))).show()

Note: Converts all letters in the Country column to lowercase. 

3. Convert all text to uppercase (upper): 

In [0]:
df.select(upper(col("Country"))).show()

Note: Converts all letters in the Country column to uppercase. 

Concatenation Functions

1. Concatenate two columns: 

In [0]:
df.select(concat(col("Region"),col("Country"))).show()

Note: Concatenates the values of Region and Country without any separator. 

2.Concatenate with a separator: 

In [0]:
df.select(concat_ws(' | ', col("Region"), col("Country"))).show()

Note: Concatenates the values of Region and Country with | as a separator.

3. Create a new concatenated column:

In [0]:
concatenated_df = df.withColumn("concatenated",concat(df["Region"],lit(" "),df["Country"])).show()

Note: This creates a new column concatenated by combining Region and Country with a space between them. 

**Summary of Key Functions:** 
-  **String Manipulation**: You can convert strings to lowercase, uppercase, or capitalize the first letter of each word. Use initcap(), lower(), and upper() for these transformations. 
-  **Concatenation**: Use concat() to join two columns or concat_ws() to join with a separator. 

These functions and transformations are common in PySpark for manipulating and querying 
data effectively!

#  Split Function In Dataframe

- Let's create a PySpark DataFrame for employee data, which will include columns such as EmployeeID, Name, Department, and Skills.  
- I'll demonstrate the usage of the split, explode, and other relevant PySpark functions with the employee data, along with notes for each operation. 

In [0]:
data = [ 
    (1, "Alice", "HR", "Communication Management"), 
    (2, "Bob", "IT", "Programming Networking"), 
    (3, "Charlie", "Finance", "Accounting Analysis"), 
    (4, "David", "HR", "Recruiting Communication"), 
    (5, "Eve", "IT", "Cloud DevOps") 
]

columns = ["EmployeeID", "Name", "Department", "Skills"] 

In [0]:
df = spark.createDataFrame(data, columns)
df.show()

1. Split the "Skills" column: 

We will split the Skills column into an array, where each skill is separated by a space.

In [0]:
df2 = df.select(col("EmployeeID"),col("Name"),split(col("skills")," ").alias("Skills_Array"))
df2.show()

Note: This splits the Skills column into an array of skills based on the space separator. The alias("Skills_Array") gives the resulting array a meaningful name. 

2. Select the first skill from the "Skills_Array": 

You can select specific elements from an array using index notation. In this case, we’ll select the first skill from the Skills_Array. 

In [0]:
df2.select(col("EmployeeID"),col("Name"),col("Skills_Array")[0].alias("first_skill")).show()


Note: The array index starts from 0, so Skills_Array[0] gives the first skill for each employee. 

3. Calculate the size of the "Skills_Array": 

We can calculate how many skills each employee has by using the size() function. 

In [0]:
df2.select(col("EmployeeID"),col("Name"),size(col("Skills_Array")).alias("Nimber_of_skills")).show()

**Note:** The size() function returns the number of elements (skills) in the Skills_Array. 

### 4. Check if the array contains a specific skill: 

We can check if a particular skill (e.g., "Cloud") is present in the employee's skillset using the array_contains() function. 

In [0]:
df.select(col("EmployeeID"),col("Name"),array_contains(split(col("skills")," ",),"Cloud").alias("has_cloud")).show()

**Note:** This returns a boolean indicating whether the array contains the specified skill,"Cloud", for each employee.

### 5. Use the explode function to transform array elements into individual rows: 

The explode() function can be used to flatten the array into individual rows, where each skill becomes a separate row for the employee. 

In [0]:
df3 = df2.withColumn("skill",explode(col("skills_Array")))
df3.select("EmployeeID","Name","Skill").show()

**Note:** The explode() function takes an array column and creates a new row for each element of the array. Here, each employee will have multiple rows, one for each skill.

**Summary of Key Functions:** 
-  **split()**: This splits a column's string value into an array based on a specified delimiter(in this case, a space). 
- **explode()**: Converts an array column into multiple rows, one for each element in the array. 
- **size()**: Returns the number of elements in an array. 
- **array_contains()**: Checks if a specific value exists in the array. 
- **selectExpr()**: Allows you to use SQL expressions (like array[0]) to select array elements. 

## Trim Function in Dataframe 

Demonstrate the usage of ltrim(), rtrim(), trim(), lpad(), and rpad() on string columns

In [0]:
data = [ 
    (1, " Alice   ", "HR"), 
    (2, "  Bob", "IT"), 
    (3, "Charlie  ", "Finance"), 
    (4, "  David ", "HR"), 
    (5, "Eve  ", "IT") 
] 
 
# Define the schema for the DataFrame 
columns = ["EmployeeID", "Name", "Department"] 
 
# Create DataFrame 
df = spark.createDataFrame(data, columns) 
 
# Show the original DataFrame 
df.show(truncate=False)

Applying Trimming and Padding Functions

**1. ltrim(), rtrim(), and trim():**

-  ltrim(): Removes leading spaces. 
-  rtrim(): Removes trailing spaces. 
-  trim(): Removes both leading and trailing spaces. 

**2. lpad() and rpad():** 
- lpad(): Pads the left side of a string with a specified character up to a certain length. 
- rpad(): Pads the right side of a string with a specified character up to a certain length.

In [0]:
# Apply trimming and padding functions 
result_df = df.select(
    col("EmployeeID"),
    col("Department"),
    ltrim(col("Name")).alias("ltrim_Name"),  # Remove leading spaces 
    rtrim(col("Name")).alias("rtrim_Name"),  # Remove trailing spaces 
    trim(col("Name")).alias("trim_Name"),    # Remove both leading and trailing spaces 
    lpad(col("Name"), 10, "X").alias("lpad_Name"),  # Left pad with "X" to make the string length 10 
    rpad(col("Name"), 10, "Y").alias("rpad_Name")   # Right pad with "Y" to make the string length 10 
)

In [0]:
# Show the resulting DataFrame 
result_df.show(truncate=False)

**Output Explanation:** 

- **ltrim_Name:** The leading spaces from the Name column are removed. 
-  **rtrim_Name**: The trailing spaces from the Name column are removed. 
-  **trim_Name**: Both leading and trailing spaces are removed from the Name column. 
-  **lpad_Name**: The Name column is padded on the left with "X" until the string length becomes 10. 
-  **rpad_Name**: The Name column is padded on the right with "Y" until the string length becomes 10. 

## Date Function in Dataframe – Part 1 

In PySpark, you can use various date functions to manipulate and analyze date and timestamp columns. Below, I'll provide a sample dataset and demonstrate key date functions like current_date, current_timestamp, date_add, date_sub, datediff, and months_between.

**Code Explanation with Notes **
- 1. Creating a Spark Session: 
-     We begin by creating a Spark session to run the PySpark operations. 

**2. Generating a DataFrame:** 
-    Using spark.range(10) creates a DataFrame with 10 rows and a single column(id) with numbers ranging from 0 to 9. 
-  Two additional columns are added: 

  -  **today:** Contains the current date using current_date(). 
  -  **now:** Contains the current timestamp using current_timestamp(). 

**3. Date Manipulation Functions:** 
- **date_add** : Adds a specified number of days to the date. 
-  **date_sub**: Subtracts a specified number of days from the date. 
-  **datediff**: Returns the difference in days between two dates. 
- **months_between**: Returns the number of months between two dates.

In [0]:
from pyspark.sql.functions import *

In [0]:
dateDF = spark.range(10).withColumn("today",current_date()).withColumn("now", current_timestamp()) 
# Show the DataFrame with today and now columns
dateDF.show(truncate=False)

**Explanation of Code and Output**

**1. current_date and current_timestamp:** 
 - current_date() gives the current date (e.g., 2024-10-12). 
 - current_timestamp() provides the current timestamp, which includes both date and time (e.g., 2024-10-12 12:34:56). 
 - These are used to create columns today and now in the DataFrame. 

**2. date_add and date_sub:** 
 - **date_sub(col("today"), 5):** Subtracts 5 days from the current date, so if today is 2024-10-12, it returns 2024-10-07.
 - **date_add(col("today"), 5):** Adds 5 days to the current date, returning 2024-10-17.

In [0]:
# Add 5 days and subtract 5 days from "today"
dateDF.select(date_sub(col("today"),5).alias("date_sub_5_days"),
              date_add(col("today"),5).alias("date_add_5_days")).show(1)

**3. datediff:** 
 - **datediff(col("week_ago"), col("today")):** Calculates the difference in days between the current date and 7 days ago (i.e., -7). 

In [0]:
#calculate the days diffrence between "today" and "week_ago" (7 days ago)
dateDF.withColumn("week_ago",date_sub(col("today"),7))\
    .select(datediff(col("week_ago"), col("today")).alias("days_diffrence")).show(1)


**4. months_between: **
- • **months_between(to_date(lit("2025-01-01")), to_date(lit("2024-01-01")):** Calculates the number of months between January 1, 2024, and January 1, 2025, which is -12 months because start_date is earlier than end_date. 

In [0]:
# calculate the number of months between two specific dates
dateDF.select(
    to_date(lit("2024-01-01")).alias("start_date"),
    to_date(lit("2025-01-01")).alias("end_date")
).select(months_between(col("start_date"),col("end_date")).alias("months_between")).show(1)

## Date Function in Dataframe – Part 2 

in PySpark, handling dates with the correct format and extracting date/time components such as year, month, day, etc., can be done with functions like **to_date,to_timestamp, year,month, dayofmonth, hour, minute, and second**. Below is a detailed explanation of how to work with date formats and extract date components.

**1. Default Date Parsing (to_date):** 
 - When using **to_date()**, the default date format is yyyy-MM-dd. 
 - If the format of the string does not match this, PySpark returns null for invalid date parsing. 

In [0]:
# Example: "2025-20-12" is invalid (20 is not a vaild month), returns null
dateDF.select(
    to_date(lit("2025-20-12")).alias("invalid_date"),
    to_date(lit("2025-12-21")).alias("valid_date")
).display()

 possibly malformed dates and want to avoid errors, use try_to_date instead (in Spark SQL or PySpark with expr):

In [0]:
dateDF.select(
    expr("try_to_date('2025-20-12')").alias("invalid_date_safe"),
    expr("try_to_date('2025-12-21')").alias("valid_date_safe")
).display()

**2. Handling Custom Date Formats:** 
- You can specify a custom date format using the to_date function by providing a format string, such as yyyy-dd-MM. 
- This allows PySpark to correctly parse the dates that deviate from the default format. 

In [0]:
dateFormat = "yyyy-dd-MM"
cleandateDf = spark.range(1).select(
    to_date(lit("2025-12-11"), dateFormat).alias("correct_format_date"),
    to_date(lit("2025-20-12"), dateFormat).alias("incorrect_format_date")
)
cleandateDf.show()

Here, "2017-12-11" will be parsed correctly since it fits yyyy-dd-MM, but 
"2017-20-12" will return null since the day (20) is out of the valid range for 
December (month 12)

**3. Handling Timestamps:** 
- You can use to_timestamp to convert strings with both date and time into a timestamp format. This is useful when working with datetime values. 
- After casting to a timestamp, you can extract various date/time components such as the year, month, day, hour, minute, and second. 

In [0]:
# Select all components (year, month, day, hour, minute, second) in a single display
cleandateDf.select(
    to_timestamp(col("correct_format_date"), dateFormat).alias("timestamp"),
    year(to_timestamp(col("correct_format_date"), dateFormat)).alias("year"),
    month(to_timestamp(col("correct_format_date"), dateFormat)).alias("month"),
    dayofmonth(to_timestamp(col("correct_format_date"), dateFormat)).alias("day"),
    hour(to_timestamp(col("correct_format_date"), dateFormat)).alias("hour"),
    minute(to_timestamp(col("correct_format_date"), dateFormat)).alias("minute"),
    second(to_timestamp(col("correct_format_date"), dateFormat)).alias("second")
).display()

**Detailed Explanation of Each Function** 

**1. to_date:**
-    Converts a string column to a date column based on the given format. If the format does not match, null is returned.

**2. to_timestamp:** 
- Converts a string column with date and time information into a timestamp,which includes both date and time. 

**3. Extracting Date Components:** 
-  **year:** Extracts the year from a date or timestamp. 
-  **month:** Extracts the month from a date or timestamp. 
- **dayofmonth:** Extracts the day of the month from a date or timestamp. 
- **hour:** Extracts the hour from a timestamp. 
- **minute:** Extracts the minute from a timestamp. 
- **second:** Extracts the second from a timestamp. 
 
**Sample Output** 
- For the input "2025-12-11" (with the format yyyy-dd-MM), you can expect the following results:

- Year: 2025 
-  Month: 12 
-  Day: 11 
-  Hour: 0 (since no time is provided) 
-  Minute: 0 


For invalid date strings (like "2025-20-12"), you will get null in the resulting DataFrame.

## Null Handling in Dataframe

Here’s an example of how you can use PySpark functions for null handling with sales data.The code includes null detection, dropping rows with nulls, filling null values, and using coalesce() to handle nulls in aggregations. I will provide the notes alongside the code.

In [0]:
# Sample data: sales data with nulls 
data = [ 
    ("John", "North", 100, None), 
    ("Doe", "East", None, 50), 
    (None, "West", 150, 30), 
    ("Alice", None, 200, 40), 
    ("Bob", "South", None, None), 
    (None, None, None, None) 
] 
columns = ["Name", "Region", "UnitsSold", "Revenue"] 
# Create DataFrame 
df = spark.createDataFrame(data, columns) 
df.show()

**Notes:** 

**1. Detecting Null Values:**

- The isNull() function identifies rows where a specified column has null values. The output shows a boolean flag for each row to indicate whether the value in the column is null.

In [0]:
# detecting null values in the region column 
df.select("Name","Region",isnull("Region").alias("is_Region_null")).show()

**2. Dropping Rows with Null Values:** 

- **dropna()** removes rows that contain null values in any column when the default mode is used. 
- Specifying "all" ensures rows are only removed if all columns contain null values. 
- You can also apply null handling only on specific columns by providing a list of column names to the subset parameter.

In [0]:
# Dropping the rows with Null values.
df2 = df.dropna()
df2.show()

In [0]:
# Dropping rows where all values are Null
df3= df.na.drop("all")
df3.show()

In [0]:
# dropping the rows if null values exixts in "Name" or "region" columns 
df4 = df.na.drop("all", subset=["Name","Region"])
df4.show()

**3. Filling Null Values:** 

- **fillna()** allows replacing null values with specified replacements, either for all columns or selectively. 
- In the example, nulls in Region are replaced with "Unknown", while UnitsSold and Revenue nulls are filled with 0. 

In [0]:
# Filling the Null Values with specific Values
df5 = df.fillna({"Region":"Unknown","UnitsSold":0,"Revenue": 0})
df5.show()

In [0]:
# Filling all Null values in "Region" and "Name" columns
df6 = df.na.fill("N/A", subset=["Name","Region"])
df6.show()

**4. Coalesce Function:**

- The **coalesce()** function returns the first non-null value in a list of columns. It’s useful when you need to handle missing data by providing alternative values from other columns. 

In [0]:
# using coalesce() to handle nulls by taking the first non-null value.
df7 = df.withColumn("Adjusted_UnitsSold",coalesce("UnitsSold","Revenue"))
df7.show()

“If **UnitsSold is not null**, use it. But if **UnitsSold is null**, then use **Revenue** instead.”

**Handling Nulls in Aggregations:**

- Null values can distort aggregate functions like mean(). Using coalesce() in an aggregation ensures that any null values are replaced with a default (e.g., 0.0) to avoid skewing the results. 

In [0]:
#Aggregating while handling null values using coalesce
df8 = df.groupBy("Region").agg(coalesce(mean("UnitsSold"),lit(0)).alias("Avg_UnitsSold"))
df8.show()

**Null Handling in DataFrames - Summary** 
- **1. Detecting Nulls:** Use isNull() to identify null values in specific columns. 
- **2. Dropping Nulls:** dropna() removes rows with null values, either in any or all columns.You can target specific columns using the subset parameter. 
- **3. Filling Nulls:** fillna() replaces nulls with specified default values, either for all or selected columns. 
- **4. Coalesce Function:**
coalesce() returns the first non-null value from multiple columns,providing a fallback when some columns contain nulls. 
- **5. Aggregations:** Use coalesce() during aggregations like mean() to handle nulls by substituting them with defaults (e.g., 0), ensuring accurate results. 

## Aggregate function in Dataframe – Part 1 

Let's create a sample DataFrame using PySpark that includes various numerical values. This dataset will be useful for demonstrating the aggregate functions. 

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
data = [ 
    Row(id=1, value=10), 
    Row(id=2, value=20), 
    Row(id=3, value=30), 
    Row(id=4, value=None), 
    Row(id=5, value=40), 
    Row(id=6, value=20) 
]

In [0]:
df = spark.createDataFrame(data)
df.show()

Aggregate Functions in PySpark 

1. Summation (sum): Sums up the values in a specified column.

In [0]:
total_sum = df.select(sum("value")).show()

2.  average of the values in a specified column.

In [0]:
#Average
average_value = df.select(avg("value")).show()

3. Count (count): Counts the number of non-null values in a specified column. 

In [0]:
#count
non_null_count = df.select(count("value")).show()

**4. Maximum (max) and Minimum (min):** Finds the maximum and minimum values in a specified column. 

In [0]:
#Maximun and Minimum
max_min_values = df.select(max("value").alias("max_value"),min("value").alias("min_value")).show()


**5. Distinct Values Count (countDistinct):** Counts the number of distinct values in a specified column. 

In [0]:
#Distinct Values Count
df.select(countDistinct("value")).show()

**Notes**
- **Handling Nulls:** The count function will count only non-null values, while sum, avg,max, and min will ignore null values in their calculations. 
- **Performance:** Aggregate functions can be resource-intensive, especially on large datasets. Using the appropriate partitioning can improve performance. 

**Use Cases:**

- **Summation:** Useful for calculating total sales, total revenue, etc. 

- **Average:** Helpful for finding average metrics like average sales per day. 

- **Count:** Useful for counting occurrences, such as the number of transactions.

- **Max/Min:** Helps to determine the highest and lowest values, such asmaximum sales on a specific day. 

- **Distinct Count:** Useful for finding unique items, like unique customers or products. 